In [ ]:
# !pip install matplotlib numpy pandas wandb x-transformers nbconvert ipykernel datasets tqdm
# !pip install ipywidgets --upgrade

In [ ]:
# !sudo apt install unzip -y

# !git clone https://github.com/fchollet/ARC.git
# !git clone https://github.com/michaelhodel/re-arc.git
# !cd re-arc && unzip re_arc.zip

# !rm -rf re-arc/.git
# !rm -rf ARC/.git

# !pip install gdown
# !gdown --folder https://drive.google.com/drive/folders/1LIc90R2MYaY_2ErAYywPZJP-SFspZt44 -O ./arc-prize-2025

In [ ]:
import sys
print(sys.version)

In [7]:
import os
from os.path import join as path_join
import json
from pathlib import Path
from functools import reduce
import matplotlib.pyplot as plt
from matplotlib import colors
from mpl_toolkits.axes_grid1 import ImageGrid
import matplotlib.patches as patches
import io
import time

import numpy as np
import pandas as pd
from PIL import Image
import datetime
import math

In [ ]:
import torch
import torch.nn.functional as F
import torch.nn as nn
from torch.optim import AdamW
from torch.utils.data import Dataset, DataLoader, TensorDataset
from torch.optim.lr_scheduler import LambdaLR

torch.set_float32_matmul_precision("high")

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(device)

In [9]:
from utils import *

def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

def prep_target_seq(input):
    # Create a new tensor with the same shape
    shifted_input = torch.empty_like(input)

    # Set the first token of every sequence to the start token
    shifted_input[:, 0] = SOS_TOKEN

    # Copy all tokens except the last one from input_examples to shifted_input_examples starting at position 1
    shifted_input[:, 1:] = input[:, :-1]

    return shifted_input

#### wanb

In [ ]:
import wandb
import os

WANDB_API_KEY = ''
os.environ['WANDB_API_KEY'] = WANDB_API_KEY
os.environ['WANDB_MODE'] = 'online' #'offline'
wandb.login()

#### Datasets

In [12]:
data_path = Path('ARC/data')
training_path = data_path / 'training'
evaluation_path = data_path / 'evaluation'

In [13]:
# size is max 30x30
# integer between 0 and 9, which are visualized as colors

def store_tasks(path, is_train = True):
  training_tasks = sorted(os.listdir(path))
  records = [] 
  for task_name in training_tasks:
    task_file = str(path / task_name)

    with open(task_file, 'r') as f:
      task = json.load(f)

      for key in task:
        if key not in ['train', 'test']:
          continue
        for t in task[key]:
          rec = {
            'task_id' : task_name, 
            'input' : t['input'], 
            'output' : t['output'],
            'final_task': key != 'train',
            'train' : is_train 
          }
          records.append(rec)

  #df = pd.DataFrame(columns=['task_id', 'input', 'output', 'final_task', 'train']) 
  df = pd.DataFrame(records)
  
  return df


df_train = store_tasks(training_path, True) 
df_test = store_tasks(evaluation_path, False)
df = pd.concat([df_train, df_test])

train_task_ids = df_train['task_id'].unique()
test_task_ids = df_test['task_id'].unique()

In [ ]:
def load_json(path):
    with open(path, 'r') as f:
        return json.load(f)
    
training_challenges_path = 'arc-prize-2025/arc-agi_training_challenges.json'
training_solutions_path = 'arc-prize-2025/arc-agi_training_solutions.json'

evaluation_challenges_path = 'arc-prize-2025/arc-agi_evaluation_challenges.json'
evaluation_solutions_path = 'arc-prize-2025/arc-agi_evaluation_solutions.json'

training_challenges = load_json(training_challenges_path)
training_solutions = load_json(training_solutions_path)

evaluation_challenges = load_json(evaluation_challenges_path)
evaluation_solutions = load_json(evaluation_solutions_path)

print(len(training_challenges), len(training_solutions), len(evaluation_challenges), len(evaluation_solutions))

def create_arc2_dataset(challenges, solutions):
    records = []
    for key, value in challenges.items():
        # save examples
        for example in value['train']:
            rec = {
                'task_id' : key, 
                'input' : example['input'], 
                'output' : example['output'],
                'final_task': False,
            }
            records.append(rec)

        # save final task
        for task_input, task_output in zip(value['test'], solutions[key]):
            rec = {
                'task_id' : key, 
                'input' : task_input['input'], 
                'output' : task_output,
                'final_task': True,
            }
            records.append(rec)

    return records

df_train_arc2 = pd.DataFrame(create_arc2_dataset(training_challenges, training_solutions))
df_test_arc2 = pd.DataFrame(create_arc2_dataset(evaluation_challenges, evaluation_solutions))

train_task_ids_arc2 = df_train_arc2['task_id'].unique()
test_task_ids_arc2 = df_test_arc2['task_id'].unique()

In [15]:
NUMBER_OF_COLORS = 10 +1 +1 # integer between 0 and 9 = 10, and +1 my class (used as background (10)) + 1 for SOS token
MAX_IMAGE_SIZE = 32
MAX_SEQUENCE_SIZE = MAX_IMAGE_SIZE * MAX_IMAGE_SIZE
BACKGROUND_CLASS = 10
SOS_TOKEN = 11

# make all images 32 by 32 and fill empty spaces by additional class
def placeholder(arg, sizes = (MAX_IMAGE_SIZE, MAX_IMAGE_SIZE), background=BACKGROUND_CLASS):
  img = np.array(arg)
  holder = np.full(sizes, background, dtype=int)
  holder[0:img.shape[0], 0:img.shape[1]] = img

  return holder

In [16]:
class CustomDatasetTestTransformationsDiscrete(Dataset):
    def __init__(self, task_ids, df, n_of_samples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.n_of_samples = n_of_samples
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        examples = self.data_by_task_id[task_id]
        selected_examples = random.sample(examples, self.n_of_samples + 1)

        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        data_item_input = [transformation_32(example['input']) for example in selected_examples]
        data_item_output = [transformation_32(example['output']) for example in selected_examples]

        return [
            torch.stack(data_item_input[:self.n_of_samples]), torch.stack(data_item_output[:self.n_of_samples]), 
            torch.stack(data_item_input[self.n_of_samples:]), torch.stack(data_item_output[self.n_of_samples:]),
        ]
    

class CustomDatasetTestTransformationsAdaptDiscrete(Dataset):
    def __init__(self, task_ids, df, n_of_samples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.n_of_samples = n_of_samples
        self.df = df
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        examples = self.df.query(f'final_task == False and task_id == "{task_id}"').to_dict(orient='records')
        selected_examples = random.sample(examples, self.n_of_samples)

        final_task = self.df.query(f'final_task == True and task_id == "{task_id}"').to_dict(orient='records')[0]

        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        data_item_input = [transformation_32(example['input']) for example in selected_examples]
        data_item_output = [transformation_32(example['output']) for example in selected_examples]
        data_item_input_d = [transformation_32(final_task['input'])]
        data_item_output_d = [transformation_32(final_task['output'])]

        return [
            torch.stack(data_item_input), torch.stack(data_item_output), 
            torch.stack(data_item_input_d), torch.stack(data_item_output_d)
        ]
    

class CustomDatasetTestTransformationsDiscreteReArc2(Dataset):
    def __init__(self, task_ids, data_points, n_of_samples=2):
        self.task_ids = task_ids
        self.data_points = data_points
        self.n_of_samples = n_of_samples
        self.task_data = self._load_all_tasks()
        
    def _load_all_tasks(self):
        """Preloads all task files into memory."""
        task_data = {}
        for task_id in self.task_ids:
            task_file = f're-arc/re_arc/tasks/{task_id}'
            with open(task_file, 'r') as f:
                examples = json.load(f)

                examples_filtered = []
                for example in examples:
                    if len(example['input']) <= 32 and len(example['input'][0]) <= 32 and len(example['output']) <= 32 and len(example['output'][0]) <= 32:
                        examples_filtered.append(example)

                task_data[task_id] = examples_filtered
        return task_data
    
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        examples = self.task_data[task_id]  # Fetch preloaded data
        
        selected_examples = random.sample(examples, self.n_of_samples+1)

        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        data_item_input = [transformation_32(example['input']) for example in selected_examples]
        data_item_output = [transformation_32(example['output']) for example in selected_examples]

        return [
            torch.stack(data_item_input[:self.n_of_samples]), torch.stack(data_item_output[:self.n_of_samples]), 
            torch.stack(data_item_input[self.n_of_samples:]), torch.stack(data_item_output[self.n_of_samples:]),
        ]

In [17]:
# # make sure arc task is not in the re-arc dataset
# def get_arc_rearc_tasks(task_id):
#     task_file = f're-arc/re_arc/tasks/{task_id}'
#     with open(task_file, 'r') as f:
#         examples = json.load(f)

#         examples_filtered = []
#         for example in examples:
#             if len(example['input']) <= 32 and len(example['input'][0]) <= 32 and len(example['output']) <= 32 and len(example['output'][0]) <= 32:
#                 examples_filtered.append(example)
#     rearc_tasks = examples_filtered


#     data = df.query(f'task_id == "{task_id}"').to_dict(orient='records')
#     arc_tasks = []
#     for example in data:
#         arc_tasks.append({
#             'input': example['input'],
#             'output': example['output'],
#         })

#     return rearc_tasks, arc_tasks

# #task_id = '0a938d79.json'
# for task_id in train_task_ids:
#     rearc_tasks, arc_tasks = get_arc_rearc_tasks(task_id)
#     #print(task_id, len(rearc_tasks), len(arc_tasks))
#     for rearc_task in rearc_tasks:
#         for arc_task in arc_tasks:
#             if rearc_task['input'] == arc_task['input']:
#                 print(rearc_task['input'], rearc_task['output'])
#                 print(arc_task['input'], arc_task['output'])
#                 print('---')

In [18]:
class CustomDatasetTestTransformationsAdaptDiscreteAll(Dataset):
    def __init__(self, task_ids, df, max_examples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.max_examples = max_examples
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        task_data = self.data_by_task_id[task_id]
        
        # Select all non-final examples for the task
        examples = [d for d in task_data if not d['final_task']]
        random.shuffle(examples)  # Shuffle examples
        examples = examples[:self.max_examples]  # Limit to max_examples


        # Get the final task example
        final_task = [d for d in task_data if d['final_task']][0]

        # Apply transformations
        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        # Process all examples
        data_item_input = [transformation_32(ex['input']) for ex in examples]
        data_item_output = [transformation_32(ex['output']) for ex in examples]
        # Process final task
        data_item_input_d = [transformation_32(final_task['input'])]
        data_item_output_d = [transformation_32(final_task['output'])]

        return [
            torch.stack(data_item_input), torch.stack(data_item_output), 
            torch.stack(data_item_input_d), torch.stack(data_item_output_d)
        ]
    

class CustomDatasetTestTransformationsDiscreteWithoutTaskAll(Dataset):
    def __init__(self, task_ids, df, max_examples=2):
        self.task_ids = task_ids
        self.data_by_task_id = {
            task_id: df.query(f'task_id == "{task_id}"').to_dict(orient='records')
            for task_id in task_ids
        }
        self.max_examples = max_examples
        
    def __len__(self):
        return len(self.task_ids)

    def __getitem__(self, idx):
        task_id = self.task_ids[idx]
        task_data = self.data_by_task_id[task_id]
        
        examples = [d for d in task_data if not d['final_task']]
        random.shuffle(examples) 
        final_task = examples.pop()
        examples = examples[:self.max_examples] 

        # Apply transformations
        transformation_1 = sample_transformations()
        transformation_2 = lambda x: placeholder(transformation_1(x))
        transformation_32 = lambda x: torch.tensor(transformation_2(x).flatten(), dtype=torch.long)

        # Process all examples
        data_item_input = [transformation_32(ex['input']) for ex in examples]
        data_item_output = [transformation_32(ex['output']) for ex in examples]
        # Process final task
        data_item_input_d = [transformation_32(final_task['input'])]
        data_item_output_d = [transformation_32(final_task['output'])]

        return [
            torch.stack(data_item_input), torch.stack(data_item_output), 
            torch.stack(data_item_input_d), torch.stack(data_item_output_d)
        ]

#### Models

In [21]:
from x_transformers import TransformerWrapper, Encoder, Decoder, PrefixDecoder
from x_transformers.autoregressive_wrapper import AutoregressiveWrapper

#### Get Predictions and Losses

In [28]:
def get_predictions_simple(model, input_examples, output_examples, input_task, output_task):
    n_examples = input_examples.shape[1]
    
    input_examples = input_examples.to(device)
    output_examples = output_examples.to(device)
    input_task = input_task.to(device)
    output_task = output_task.to(device)

    input_examples = input_examples.view(-1, MAX_SEQUENCE_SIZE)
    input_examples_emb = model.encode(input_examples)
    input_examples_emb = input_examples_emb.view(-1, n_examples, MAX_SEQUENCE_SIZE, input_examples_emb.shape[-1])

    output_examples = output_examples.view(-1, MAX_SEQUENCE_SIZE)
    output_examples_emb = model.encode(output_examples)
    output_examples_emb = output_examples_emb.view(-1, n_examples, MAX_SEQUENCE_SIZE, output_examples_emb.shape[-1])

    input_task = input_task.view(-1, MAX_SEQUENCE_SIZE)
    input_task_emb = model.encode(input_task)
    input_task_emb = input_task_emb.view(-1, 1, MAX_SEQUENCE_SIZE, input_task_emb.shape[-1])

    output_task = output_task.view(-1, MAX_SEQUENCE_SIZE)
    output_task_emb = model.encode(output_task)
    output_task_emb = output_task_emb.view(-1, 1, MAX_SEQUENCE_SIZE, output_task_emb.shape[-1])

    # ----------------------------
    c = (output_examples_emb - input_examples_emb).mean(dim=(1, 2), keepdim=True)
    
    # calculate embedding losses
    input_examples_emb_est = output_examples_emb - c
    output_examples_emb_est = input_examples_emb + c

    input_task_emb_est = output_task_emb - c
    output_task_emb_est = input_task_emb + c

    em_dim = model.dim
    loss_reconstracted_emb_1 = F.mse_loss(input_examples_emb_est.view(-1, em_dim), input_examples_emb.view(-1, em_dim))
    loss_reconstracted_emb_2 = F.mse_loss(output_examples_emb_est.view(-1, em_dim), output_examples_emb.view(-1, em_dim))
    loss_reconstracted_emb_3 = F.mse_loss(input_task_emb_est.view(-1, em_dim), input_task_emb.view(-1, em_dim))
    loss_reconstracted_emb_4 = F.mse_loss(output_task_emb_est.view(-1, em_dim), output_task_emb.view(-1, em_dim))
    loss_reconstracted_emb = loss_reconstracted_emb_1 + loss_reconstracted_emb_2 + loss_reconstracted_emb_3 + loss_reconstracted_emb_4

    # calculate reconstruction losses
    em_dim = model.dim
    input_examples_est_reconstracted = model.decode(input_examples, input_examples_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))
    output_examples_est_reconstracted = model.decode(output_examples, output_examples_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))
    input_task_est_reconstracted = model.decode(input_task, input_task_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))
    output_task_est_reconstracted = model.decode(output_task, output_task_emb_est.view(-1, MAX_SEQUENCE_SIZE, em_dim))

    loss_reconstracted_img_1 = F.cross_entropy(input_examples_est_reconstracted.view(-1, NUMBER_OF_COLORS), input_examples.view(-1))
    loss_reconstracted_img_2 = F.cross_entropy(output_examples_est_reconstracted.view(-1, NUMBER_OF_COLORS), output_examples.view(-1))
    loss_reconstracted_img_3 = F.cross_entropy(input_task_est_reconstracted.view(-1, NUMBER_OF_COLORS), input_task.view(-1))
    loss_reconstracted_img_4 = F.cross_entropy(output_task_est_reconstracted.view(-1, NUMBER_OF_COLORS), output_task.view(-1))
    loss_reconstracted_img = loss_reconstracted_img_1 + loss_reconstracted_img_2 + loss_reconstracted_img_3 + loss_reconstracted_img_4

    loss = 0.01 * loss_reconstracted_emb + loss_reconstracted_img
    #loss += 0.1 * torch.mean(torch.abs(c))
    loss += 0.1 * torch.mean(c.pow(2))

    loss_log = {
        'total_loss': loss.item(),
        'loss_reconstracted_img_1': loss_reconstracted_img_1.item(),
        'loss_reconstracted_img_2': loss_reconstracted_img_2.item(),
        'loss_reconstracted_img_3': loss_reconstracted_img_3.item(),
        'loss_reconstracted_img_4': loss_reconstracted_img_4.item(),
        'loss_reconstracted_img': loss_reconstracted_img.item(),
        
        'loss_reconstracted_emb': loss_reconstracted_emb.item(),
        'loss_reconstracted_emb_1': loss_reconstracted_emb_1.item(),
        'loss_reconstracted_emb_2': loss_reconstracted_emb_2.item(),
        'loss_reconstracted_emb_3': loss_reconstracted_emb_3.item(),
        'loss_reconstracted_emb_4': loss_reconstracted_emb_4.item(),
    }

    emb_data = (input_examples_emb_est, output_examples_emb_est, input_task_emb_est, output_task_emb_est)
    decoded_data = (input_examples_est_reconstracted, output_examples_est_reconstracted, input_task_est_reconstracted, output_task_est_reconstracted)

    return loss, emb_data, decoded_data, loss_log

### GPT

In [37]:
import torch
from x_transformers import TransformerWrapper, Decoder
from torch.optim import AdamW
from tqdm.auto import tqdm

In [38]:
def count_parameters(model):
    return sum(p.numel() for p in model.parameters() if p.requires_grad)

In [39]:
import torch.nn as nn
from x_transformers import TransformerWrapper, Decoder

class GPTArc(nn.Module):
    def __init__(self, num_tokens=NUMBER_OF_COLORS, max_seq_len=MAX_SEQUENCE_SIZE * 4, dim=64, depth=4, heads=8):
        super().__init__()

        self.decoder = TransformerWrapper(
            num_tokens=num_tokens,
            max_seq_len=max_seq_len,
            attn_layers=Decoder(
                dim=dim,
                depth=depth,
                heads = heads
            )
        )

    def forward(self, x):
        return self.decoder(x)

In [40]:
def evaluate(model, data_loader, device):
    criterion = torch.nn.CrossEntropyLoss()

    model.eval()
    total_loss = 0
    total_correct = 0
    total_tokens = 0
    total_all_correct = 0

    with torch.no_grad():
        for input_examples, output_examples, input_task, output_task in data_loader:
            prompt = torch.stack([input_examples[:, :1, :], output_examples[:, :1, :], input_task, output_task], dim=1)
            prompt = prompt.view(-1, 4 * MAX_SEQUENCE_SIZE).to(device)
            output_task = output_task.to(device)

            with torch.autocast(device_type=str(device), dtype=torch.bfloat16):
                logits = model(prep_target_seq(prompt))

                logits_for_output_task = logits[:, -MAX_SEQUENCE_SIZE:, :]
                logits_flat = logits_for_output_task.reshape(-1, logits.size(-1))
                targets_flat = output_task.reshape(-1)

                # logits_flat = logits.reshape(-1, logits.size(-1))
                # targets_flat = prompt.reshape(-1)

                loss = criterion(logits_flat, targets_flat)
                total_loss += loss.item()

                # ---------- TOKEN ACC ----------
                predictions = torch.argmax(logits_flat, dim=-1)
                correct = (predictions == targets_flat).sum().item()
                total_correct += correct
                total_tokens += targets_flat.numel()

                # ---------- ALL-CORRECT ----------
                preds_seq = predictions.view(output_task.size(0), -1)
                targets_seq = targets_flat.view(output_task.size(0), -1)

                # A task is correct if ALL tokens match
                all_correct_batch = (preds_seq == targets_seq).all(dim=1).sum().item()
                total_all_correct += all_correct_batch

    avg_loss = total_loss / len(data_loader)
    accuracy = total_correct / total_tokens
    return avg_loss, accuracy, total_all_correct

In [41]:
def get_predictions_simple(model, input_examples, output_examples, input_task, output_task, device=device):
    criterion = torch.nn.CrossEntropyLoss()
    
    prompt = torch.stack(
        [input_examples[:, :1, :], output_examples[:, :1, :], input_task, output_task],
        dim=1
    )
    prompt = prompt.view(-1, 4 * MAX_SEQUENCE_SIZE).to(device)
    output_task = output_task.to(device)


    logits = model(prep_target_seq(prompt))
    
    # logits_for_output_task = logits[:, -MAX_SEQUENCE_SIZE:, :]
    # logits_flat = logits_for_output_task.reshape(-1, logits.size(-1))
    # targets_flat = output_task.reshape(-1)

    logits_flat = logits.reshape(-1, logits.size(-1))
    targets_flat = prompt.reshape(-1)

    loss = criterion(logits_flat, targets_flat)

    return loss

#### TTT GPT

In [42]:
def run_eval_ttt_gpt_adaptor(model, loader):
    ttt_loss, ttt_accuracy, ttt_all_correct = evaluate(model, loader, device)
    summary = {
        'all_correct_img_4': ttt_all_correct,
        'avg_proportion_correct_img_4': ttt_accuracy,
        'total_loss': ttt_loss
    }

    return summary

In [43]:
def run_fine_tune(model, optimizer, train_loader, test_loader, n_epoch_finetune, grad_accum_steps=1):
    num_warmup_batches = 4  # Set the number of warm-up batches
    def lr_lambda(current_step):
        if current_step < num_warmup_batches:
            return float(current_step) / float(max(1, num_warmup_batches))
        return 1.0
    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    for epoch in range(n_epoch_finetune):
        model.train()
        total_loss = 0

        for batch_idx, (input_examples, output_examples, input_task, output_task) in enumerate(train_loader):
            with torch.autocast(device_type=str(device), dtype=torch.bfloat16):
                loss = get_predictions_simple(model, input_examples, output_examples, input_task, output_task)
                loss /= grad_accum_steps

            # Accumulating gradients over multiple steps
            loss.backward()  # Retain graph for reuse during accumulation
            
            # Check if it's time to update the weights
            if (batch_idx + 1) % grad_accum_steps == 0:
                # Clip gradients to avoid exploding gradients
                torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
                
                # Update weights
                optimizer.step()
                scheduler.step()
                
                # Zero gradients for the next accumulation step
                optimizer.zero_grad()

            total_loss += loss.item()

        with torch.no_grad():
            total_loss = total_loss / len(train_loader) * grad_accum_steps 
            print(f"Epoch {epoch + 1} completed with total loss: {total_loss}", )
            
            # model.eval()
            # train_data = run_eval(model, train_loader)
            # test_data = run_eval(model, test_loader)
            # print(f"Epoch {epoch + 1} completed with total loss: {total_loss}", 'Train:', train_data['avg_proportion_correct_img_4'], train_data['all_correct_img_4'], train_data['total_loss_reconstracted_img_4'], 'Test:', test_data['avg_proportion_correct_img_4'], test_data['all_correct_img_4'], test_data['total_loss_reconstracted_img_4'])


        #     # # log images
        #     img_train_name, img_test_name = store_test_as_image(model, train_loader, test_loader, epoch=epoch)
        #     wandb.log({"train_images": wandb.Image(img_train_name), "test_images": wandb.Image(img_test_name)})
    

    return model

In [ ]:
def ttt_pipeline(config, model_name, run_id=0, n_tasks=None, use_wandb=True):
    if os.path.exists(model_name):
        print("Model exists!", model_name)
    else:
        print("File does NOT exist!", model_name)
        return
    
    if n_tasks is None:
        n_tasks = 400        # full evaluation
    else:
        n_tasks = int(n_tasks)       # e.g., for fast mode (40)

    debug_task_ids = test_task_ids # train_task_ids
    debug_df = df_test #df_train

    em_dim = 256
    depth = 4
    model = GPTArc(dim=em_dim, depth=depth, heads=8).to(device)
    #model = torch.compile(model)

    n_epoch_finetune = 2
    n_train_set_finetune = 256
    n_test_set = 256
    bs_finetune = 8
    bs_test = bs_finetune * 2
    grad_accum_steps = 1
    learning_rate = 3e-4

    if use_wandb:
        wandb.init(project="eval_ttt_erd", 
                name=f"{config['NAME']}_run_{run_id}",
                    group=config["NAME"],
                config={
            "learning_rate": learning_rate,
            "batch_size": bs_finetune*grad_accum_steps,
            "em_dim": em_dim,
            "depth": depth,
            "n_epoch_finetune": n_epoch_finetune,
            "n_train_set_finetune": n_train_set_finetune,
            "model_size": count_parameters(model)
        })

    n_correct_per_task_before = []
    n_correct_per_task_after = []
    tasks_with_zero_solved_rate = []

    #for task_id in [4]:
    for task_id in range(n_tasks):
        # clear cache
        torch.cuda.empty_cache()

        print(f'Progress {task_id} over {n_tasks}:')
        # load model
        model.load_state_dict(torch.load(model_name, weights_only=True))
        optimizer = AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95))
        
        # train loader
        test_task_ids_single = [debug_task_ids[task_id]] * n_train_set_finetune
        dataset_train = CustomDatasetTestTransformationsDiscreteWithoutTaskAll(test_task_ids_single, debug_df, max_examples=2)
        train_adaptation_loader = DataLoader(dataset_train, batch_size=bs_finetune, shuffle=True, drop_last=True)
        # eval loaders
        test_task_ids_single = [debug_task_ids[task_id]] * n_test_set
        dataset_test_eval = CustomDatasetTestTransformationsAdaptDiscreteAll(test_task_ids_single, debug_df, max_examples=2)
        eval_adaptation_loader = DataLoader(dataset_test_eval, batch_size=bs_test, shuffle=True, drop_last=True)

        # evaluate before fine-tuning
        with torch.no_grad():
            model.eval()
            train_data = run_eval_ttt_gpt_adaptor(model, train_adaptation_loader)
            test_data = run_eval_ttt_gpt_adaptor(model, eval_adaptation_loader)
            n_correct_per_task_before.append(test_data['all_correct_img_4'] / n_test_set)
            
            print('Before.', 
                'Train:', 
                'avg prop correct px =', f"{train_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{train_data['all_correct_img_4'] / n_train_set_finetune :.4f}", 
                'Test:', 
                'avg prop correct px =', f"{test_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{test_data['all_correct_img_4'] / n_test_set:.4f}")

        # fine tune
        model = run_fine_tune(model, optimizer, train_adaptation_loader, eval_adaptation_loader, n_epoch_finetune, grad_accum_steps)

        # max_runs = 10
        # run_count = 0
        # stop_when_n_solved = 200
        # while True:
        #     with torch.no_grad():
        #         model.eval()
        #         train_data = run_eval(model, train_adaptation_loader)
        #         test_data = run_eval(model, eval_adaptation_loader)
        #         print(f'Attempt {run_count + 1}/{max_runs},', train_data['avg_proportion_correct_img_4'], train_data['all_correct_img_4'], test_data['avg_proportion_correct_img_4'], test_data['all_correct_img_4'])

        #     if train_data['all_correct_img_4'] > stop_when_n_solved or run_count >= max_runs:
        #         break

        #     model = run_fine_tune(model, optimizer, train_adaptation_loader, n_epoch_finetune)
        #     run_count += 1

        # evaluate after fine-tuning
        with torch.no_grad():
            model.eval()
            train_data = run_eval_ttt_gpt_adaptor(model, train_adaptation_loader)
            test_data = run_eval_ttt_gpt_adaptor(model, eval_adaptation_loader)
            n_correct_per_task_after.append(test_data['all_correct_img_4'] / n_test_set)

            print('After.', 
                'Train:', 
                'avg prop correct px =', f"{train_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{train_data['all_correct_img_4'] / n_train_set_finetune :.4f}", 
                'Test:', 
                'avg prop correct px =', f"{test_data['avg_proportion_correct_img_4']:.4f}", 
                'all correct =', f"{test_data['all_correct_img_4'] / n_test_set:.4f}")

        if n_correct_per_task_after[-1] == 0:
            tasks_with_zero_solved_rate.append(debug_task_ids[task_id])
        
        
        print(f'Current task. Before = {n_correct_per_task_before[-1]:.4f}, After = {n_correct_per_task_after[-1]:.4f}')  
        print(f'All tasks mean. Before = {np.mean(n_correct_per_task_before):.4f}, After = {np.mean(n_correct_per_task_after):.4f}')
        print(f'All tasks std. Before = {np.std(n_correct_per_task_before):.4f}, After = {np.std(n_correct_per_task_after):.4f}')
        print('Zero solved rate: ', len(tasks_with_zero_solved_rate), tasks_with_zero_solved_rate)
        print()


        if use_wandb:
            # log
            wandb.log({
                'proportion_solved_before': np.mean(n_correct_per_task_before),
                'proportion_solved_after': np.mean(n_correct_per_task_after),
                'proportion_solved_before_var': np.std(n_correct_per_task_before),
                'proportion_solved_after_var': np.std(n_correct_per_task_after),
                'zero_solved_rate': len(tasks_with_zero_solved_rate),
                'proportion_with_solution': ((task_id + 1) - len(tasks_with_zero_solved_rate)) / (task_id + 1),
                'proportion_with_no_solution': len(tasks_with_zero_solved_rate) / (task_id + 1)
            })

    # # log summary
    # wandb.summary['final_proportion_solved_before'] = np.mean(n_correct_per_task_before)
    # wandb.summary['final_proportion_solved_after'] = np.mean(n_correct_per_task_after)
    # wandb.summary['final_proportion_solved_before_var'] = np.std(n_correct_per_task_before)
    # wandb.summary['final_proportion_solved_after_var'] = np.std(n_correct_per_task_after)
    # wandb.summary['final_zero_solved_rate'] = len(tasks_with_zero_solved_rate)
    # wandb.summary['final_proportion_with_solution'] = (400 - len(tasks_with_zero_solved_rate)) / 400

    # wandb.finish()

    # --- final summary ---
    summary = {
        "final_proportion_solved_before": np.mean(n_correct_per_task_before),
        "final_proportion_solved_after": np.mean(n_correct_per_task_after),
        "final_proportion_solved_before_var": np.std(n_correct_per_task_before),
        "final_proportion_solved_after_var": np.std(n_correct_per_task_after),
        "final_zero_solved_rate": len(tasks_with_zero_solved_rate),
        "final_proportion_with_solution": (n_tasks - len(tasks_with_zero_solved_rate)) / n_tasks,
        "final_proportion_with_no_solution": len(tasks_with_zero_solved_rate) / n_tasks
    }

    if use_wandb:
        for k, v in summary.items():
            wandb.summary[f"final_{k}"] = v
        wandb.finish()

    return summary

In [ ]:
# model_name = "model_gpt_epoch_10000.pt"
# config = {"NAME": 'final_gpt_epoch_10000'}
# ttt_summary = ttt_pipeline(config, model_name)
# print('TTT Summary:', ttt_summary)

# raise ValueError("Stop here")

#### Multy GPU

In [ ]:
batch_size_train = 2

train_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(train_task_ids, df_train), batch_size=batch_size_train*2)
test_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(test_task_ids, df_test), batch_size=batch_size_train*2)
train_arc2_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(train_task_ids_arc2, df_train_arc2), batch_size=batch_size_train*2)
test_arc2_eval_loader_transformations = DataLoader(CustomDatasetTestTransformationsAdaptDiscrete(test_task_ids_arc2, df_test_arc2), batch_size=batch_size_train*2)

def run_eval_ddp(model, epoch):
    model.eval()
    with torch.no_grad():
        train_loss, train_accuracy, train_all_correct = evaluate(model, train_eval_loader_transformations, device)
        print(f"Epoch {epoch + 1}: Train Loss = {train_loss:.4f}, Accuracy = {train_accuracy:.4%}, All Correct = {train_all_correct}")
        
        test_loss, test_accuracy, test_all_correct  = evaluate(model, test_eval_loader_transformations, device)
        print(f"Epoch {epoch + 1}: Test Loss = {test_loss:.4f}, Accuracy = {test_accuracy:.4%}, All Correct = {test_all_correct}")

        train_arc2_loss, train_arc2_accuracy, train_arc2_all_correct = evaluate(model, train_arc2_eval_loader_transformations, device)
        print(f"Epoch {epoch + 1}: Train ARC2 Loss = {train_arc2_loss:.4f}, Accuracy = {train_arc2_accuracy:.4%}, All Correct = {train_arc2_all_correct}")
        
        test_arc2_loss, test_arc2_accuracy, test_arc2_all_correct  = evaluate(model, test_arc2_eval_loader_transformations, device)
        print(f"Epoch {epoch + 1}: Test ARC2 Loss = {test_arc2_loss:.4f}, Accuracy = {test_arc2_accuracy:.4%}, All Correct = {test_arc2_all_correct}")

    wandb.log({
        'train_total_loss_reconstracted_img_4': train_loss,
        'train_avg_proportion_correct_img_4': train_accuracy,
        'train_all_correct_img_4': train_all_correct,


        'test_total_loss_reconstracted_img_4': test_loss,
        'test_avg_proportion_correct_img_4': test_accuracy,
        'test_all_correct_img_4': test_all_correct,

        'train_arc_2_total_loss_reconstracted_img_4': train_arc2_loss,
        'train_arc_2_avg_proportion_correct_img_4': train_arc2_accuracy,
        'train_arc_2_all_correct_img_4': train_arc2_all_correct,


        'test_arc_2_total_loss_reconstracted_img_4': test_arc2_loss,
        'test_arc_2_avg_proportion_correct_img_4': test_arc2_accuracy,
        'test_arc_2_all_correct_img_4': test_arc2_all_correct
    }, step=epoch)    

    model.train()

In [ ]:
import torch
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import torch.multiprocessing as mp
from torch.utils.data.distributed import DistributedSampler
from torch.nn.parallel import DistributedDataParallel as DDP
from torch.distributed import init_process_group, destroy_process_group
import os
import copy


def ddp_setup():
    torch.cuda.set_device(int(os.environ["LOCAL_RANK"]))
    init_process_group(backend="nccl")

def ddp_average(value: torch.Tensor):
    avg = value.detach().clone()
    torch.distributed.all_reduce(avg, op=torch.distributed.ReduceOp.SUM)
    avg /= torch.distributed.get_world_size()
    return avg

class Trainer:
    def __init__(
        self,
        model: torch.nn.Module,
        train_data: DataLoader,
        optimizer: torch.optim.Optimizer,
        save_every: int,
        snapshot_path: str,
    ) -> None:
        self.gpu_id = int(os.environ["LOCAL_RANK"])
        self.model = model.to(self.gpu_id)
        self.train_data = train_data
        self.optimizer = optimizer
        self.save_every = save_every
        self.epochs_run = 0
        self.snapshot_path = snapshot_path

        # if os.path.exists(snapshot_path):
        #     print("Loading snapshot")
        #     self._load_snapshot(snapshot_path)

        self.model = DDP(self.model, device_ids=[self.gpu_id])

    def _load_snapshot(self, snapshot_path):
        loc = f"cuda:{self.gpu_id}"
        snapshot = torch.load(snapshot_path, map_location=loc)
        self.model.load_state_dict(snapshot)
        print(f"Resuming training from snapshot at Epoch {self.epochs_run}")

    # def _run_batch(self, input_examples, output_examples, input_task, output_task):
    #     self.optimizer.zero_grad()

    #     with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
    #         loss = get_predictions_simple(self.model, input_examples, output_examples, input_task, output_task, device=self.gpu_id)
    #     #print('Batch Loss:', loss.item())

    #     loss.backward()
    #     # ---- Gradient clipping ----
    #     torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
    #     self.optimizer.step()

    # def _run_epoch(self, epoch):
    #     b_sz = len(next(iter(self.train_data))[0])
    #     current_lr = self.optimizer.param_groups[0]["lr"]

    #     print(f"[GPU{self.gpu_id}] Epoch {epoch} | Batchsize: {b_sz} | Learning rate: {current_lr} | Steps: {len(self.train_data)}")
    #     self.train_data.sampler.set_epoch(epoch)
    #     for input_examples, output_examples, input_task, output_task in self.train_data:
    #         self._run_batch(input_examples, output_examples, input_task, output_task)

    def _run_epoch(self, epoch):
        b_sz = len(next(iter(self.train_data))[0])
        current_lr = self.optimizer.param_groups[0]["lr"]

        print(f"[GPU{self.gpu_id}] Epoch {epoch} | Batchsize: {b_sz} | LR: {current_lr} | Steps: {len(self.train_data)}")

        # Important for DistributedSampler
        self.train_data.sampler.set_epoch(epoch)

        epoch_loss = 0.0
        num_batches = 0

        for input_examples, output_examples, input_task, output_task in self.train_data:

            self.optimizer.zero_grad()

            with torch.autocast(device_type="cuda", dtype=torch.bfloat16):
                loss = get_predictions_simple(
                    self.model,
                    input_examples,
                    output_examples,
                    input_task,
                    output_task,
                    device=self.gpu_id
                )

            loss.backward()
            torch.nn.utils.clip_grad_norm_(self.model.parameters(), max_norm=1.0)
            self.optimizer.step()

            epoch_loss += loss.detach()
            num_batches += 1

        # ---- Local loss (this GPU only) ----
        local_epoch_loss = epoch_loss / num_batches

        # ---- Global averaged loss across all GPUs ----
        global_epoch_loss = ddp_average(local_epoch_loss)

        # ---- Log only from GPU 0 ----
        if self.gpu_id == 0:
            wandb.log({
                "total_train_loss": global_epoch_loss.item(),
                "lr": current_lr,
                "epoch": epoch
            }, step=epoch)

        # ---- Print losses (every GPU prints its own local + global) ----
        print(
            f"[GPU{self.gpu_id}] Epoch {epoch} | "
            f"Local loss: {local_epoch_loss.item():.4f} | "
            f"Global loss: {global_epoch_loss.item():.4f}"
        )

    def _save_snapshot(self, epoch):
        snapshot = self.model.module.state_dict()

        # Example: snapshot.pt → snapshot_epoch_10.pt
        epoch_path = self.snapshot_path.replace(".pt", f"_epoch_{epoch}.pt")

        torch.save(snapshot, epoch_path)
        print(f"Epoch {epoch} | Training snapshot saved at {epoch_path}")

        wandb.save(epoch_path, policy="now")

    def train(self, max_epochs: int, scheduler=None):
        for epoch in range(self.epochs_run, max_epochs):
            self._run_epoch(epoch)

            # step scheduler ONCE per epoch (safe for DDP)
            if scheduler is not None:
                scheduler.step()

            # ---- Evaluation every N epochs ----
            if self.gpu_id == 0 and (epoch % 200 == 0):
                eval_model = self.model.module
                run_eval_ddp(eval_model, epoch)

            if self.gpu_id == 0 and epoch % self.save_every == 0:
                self._save_snapshot(epoch)


def load_train_objs(config):
    train_set = CustomDatasetTestTransformationsDiscreteReArc2(train_task_ids, df_train)

    model = GPTArc(dim=config["em_dim"], depth=config["depth"], heads=config["heads"])

    learning_rate = config["learning_rate"]
    optimizer = AdamW(model.parameters(), lr=learning_rate, betas=(0.9, 0.95))

    # ---- Warmup scheduler ----
    num_warmup_epochs = config["warmup_epochs"]
    def lr_lambda(current_epoch):
        if current_epoch < num_warmup_epochs:
            return float(current_epoch + 1) / float(num_warmup_epochs)
        return 1.0
    scheduler = LambdaLR(optimizer, lr_lambda=lr_lambda)

    return train_set, model, optimizer, scheduler



def prepare_dataloader(dataset: Dataset, batch_size: int):
    return DataLoader(
        dataset,
        batch_size=batch_size,
        pin_memory=True,
        drop_last=True,
        shuffle=False,
        sampler=DistributedSampler(dataset)
    )


def main(config):
    ddp_setup()

    dataset, model, optimizer, scheduler = load_train_objs(config)
    train_data = prepare_dataloader(dataset, config["batch_size"])
    trainer = Trainer(model, train_data, optimizer, config["save_every"], config["snapshot_path"])
    
    # Only GPU0 initializes wandb
    if int(os.environ["LOCAL_RANK"]) == 0:
        # --- create copy ---
        wandb_config = copy.deepcopy(config)
        # --- update only in the copy ---
        wandb_config["batch_size"] = wandb_config["batch_size"] * torch.distributed.get_world_size()
        # Optional: log model size
        wandb_config["model_size"] = count_parameters(model)

        wandb.init(
            project=config["project"],
            name=config["run_name"],
            group=config["group"],
            config=wandb_config
        )
    
    trainer.train(config["max_epochs"], scheduler)

    destroy_process_group()


if __name__ == "__main__":
    config = {
        "project": "arc_paper_4",
        "run_name": "gpt",
        "group": "gpt",

        "em_dim": 256,
        "depth": 4,
        "heads": 8,
        
        "max_epochs": 1000000,
        "save_every": 10000,

        "batch_size": 2,
        "learning_rate": 3e-4, # update by multiply x2 if 8 gpu, x4 if 16 gpus
        "warmup_epochs": 512,
        "snapshot_path": "model_gpt.pt",
    }


    main(config)